In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats
from scipy.special import expit

In [2]:
TASK = Path.cwd().parent
ROOT = TASK.parent
INPUT, OUTPUT = TASK / 'input', TASK / 'output'
CONFIG = json.loads((ROOT / 'config.json').read_text())
EPS, Y, SEED = CONFIG['eps'], CONFIG['last_year'], CONFIG['seed']

In [3]:
N = CONFIG['n_sim']
rng = np.random.default_rng(SEED)

In [4]:
TAG = 'ar1_s'
GENERATED_FROM = 1

In [5]:
params = pd.read_csv(INPUT / TAG / 'parameters.csv', dtype={'stage':str})
panel = pd.read_csv(INPUT / 'data/panel.csv', index_col='dblp_id')
DEST = OUTPUT / TAG
DEST.mkdir(exist_ok=True)
sim = np.zeros((Y+1, N))

In [6]:
initial = pd.read_csv(INPUT / TAG / 'initialization.csv').iloc[0]

In [7]:
sim[0] = rng.exponential(initial.scale, N)
z = np.log(sim[0]+EPS)
for row in params.itertuples():
    for t in range(row.start, row.end+1):
        z_next = row.intercept + row.beta*z + rng.normal(0, row.sigma, N)
        sim[t+1] = np.maximum(np.exp(z_next)-EPS, 0)
        z = np.log(sim[t+1]+EPS)

In [8]:
np.save(DEST / 'trajectories.npy', sim, allow_pickle=False)
pd.DataFrame([{'model':TAG, 'seed':SEED, 'n_sim':N, 'n_years':Y+1, 'generated_from_age':GENERATED_FROM,
               'orientation':'age x scholar', 'minimum':sim.min(), 'maximum':sim.max(),
               'finite':np.isfinite(sim).all()}]).to_csv(DEST / 'metadata.csv', index=False)
display(pd.read_csv(DEST / 'metadata.csv'))

,model,seed,n_sim,n_years,generated_from_age,orientation,minimum,maximum,finite
0,ar1_s,63,10000,21,1,age x scholar,0.0,800.991956,True
